<a href="https://colab.research.google.com/github/Glaze0/Assignment/blob/main/Code_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Case Study: AI Meeting Action-Item Extractor
# Problem Statement

# A company records meeting notes as unstructured text. They want to use Gemini to automatically convert those notes into structured data that can be pushed into their project-management system.

# The AI should extract:

# Meeting title
# Participants
# Decisions made
# Action items
# Owner of each action item
# Due date
# Overall meeting status

# The challenge is that meeting notes are unstructured, while the downstream project-management system requires a fixed and predictable format.

# Your task is to design the Input-Output Contract before implementing the AI solution.

In [ ]:
# Sample input :
# meeting_notes = """
# Product Launch Planning Meeting

# Participants: Akash, Priya, Rahul, Neha

# The team agreed to launch the new AI resume screener
# on September 15.

# Priya will finalize the landing page by September 5.

# Rahul needs to complete the payment integration by
# September 8.

# Neha will prepare the launch email campaign by
# September 10.

# Akash will review the final product demo on September 12.

# The team decided that the launch date is confirmed.
# """

In [ ]:
# Sample output :
# {
#   "meeting_title": "Product Launch Planning Meeting",
#   "participants": [
#     "Akash",
#     "Priya",
#     "Rahul",
#     "Neha"
#   ],
#   "decisions": [
#     "Launch the AI resume screener on September 15"
#   ],
#   "action_items": [
#     {
#       "task": "Finalize the landing page",
#       "owner": "Priya",
#       "due_date": "September 5"
#     },
#     {
#       "task": "Complete the payment integration",
#       "owner": "Rahul",
#       "due_date": "September 8"
#     },
#     {
#       "task": "Prepare the launch email campaign",
#       "owner": "Neha",
#       "due_date": "September 10"
#     },
#     {
#       "task": "Review the final product demo",
#       "owner": "Akash",
#       "due_date": "September 12"
#     }
#   ],
#   "meeting_status": "Confirmed"
# }

In [ ]:
# The Problem

# Gemini might instead return:

# Priya - landing page - Sep 5
# Rahul - payment integration - Sep 8
# Neha - email campaign - Sep 10
# Akash - product demo - Sep 12

# Or:

# {
#   "tasks": [
#     "Priya: Landing page",
#     "Rahul: Payment",
#     "Neha: Email"
#   ]
# }

# Or:

# {
#   "action_items": "Priya should finish landing page..."
# }

# All contain useful information, but your Python application cannot reliably depend on these formats.

# Student Tasks

# Task 1 — Identify the Contract
# What should the input to the AI system look like?

# Input
# ----------------
# meeting_notes → ?

# Decide the:

# Field name
# Data type
# Required/optional status

# Task 2 — Design the Output Contract
# Define:

# meeting_title
# participants
# decisions
# action_items
# meeting_status

# For each field, determine:

# Data type
# Required or optional
# Array or single value

# Task 3 — Design the Nested Contract
# The difficult part is action_items.

# Each action item should contain:

# task
# owner
# due_date

# What should the data structure look like?

# action_items
#     ↓
#    Array
#     ↓
# Object
#  ├── task
#  ├── owner
#  └── due_date
# Task 4 — Define Allowed Values

# Should meeting_status accept anything?

# Or should it be restricted to:

# Confirmed
# Tentative
# Cancelled

# Similarly, think about whether other fields should have constraints.

# Task 5 — Design the Contract

# Before writing any Gemini code, complete:

# INPUT
# --------------------------
# meeting_notes → ?


# OUTPUT
# --------------------------
# meeting_title → ?
# participants → ?
# decisions → ?
# action_items → ?

# action_items:
#     task → ?
#     owner → ?
#     due_date → ?

# meeting_status → ?

In [ ]:
# ============================================================
# BLOCK 1 CASE STUDY — MEETING ACTION-ITEM EXTRACTOR
# Full Solution
# ============================================================

from google import genai
from google.genai import types
import json


API_KEY = "xx"

client = genai.Client(api_key=API_KEY)

In [ ]:
# ============================================================
# 1. INPUT
# ============================================================

meeting_notes = """
Product Launch Planning Meeting

Participants: Akash, Priya, Rahul, Neha

The team agreed to launch the new AI resume screener
on September 15.

Priya will finalize the landing page by September 5.

Rahul needs to complete the payment integration by
September 8.

Neha will prepare the launch email campaign by
September 10.

Akash will review the final product demo on September 12.

The team decided that the launch date is confirmed.
"""


# ============================================================
# 2. PROMPT
# ============================================================

prompt = f"""
Extract structured information from the following
meeting notes.

MEETING NOTES:
{meeting_notes}

Extract:
- Meeting title
- Participants
- Decisions
- Action items
- Owner and due date for each action item
- Meeting status

Do not invent information.
"""


# ============================================================
# 3. OUTPUT SCHEMA
# ============================================================

response_schema = {

    "type": "OBJECT",

    "properties": {

        "meeting_title": {
            "type": "STRING"
        },

        "participants": {
            "type": "ARRAY",
            "items": {
                "type": "STRING"
            }
        },

        "decisions": {
            "type": "ARRAY",
            "items": {
                "type": "STRING"
            }
        },

        "action_items": {

            "type": "ARRAY",

            "items": {

                "type": "OBJECT",

                "properties": {

                    "task": {
                        "type": "STRING"
                    },

                    "owner": {
                        "type": "STRING"
                    },

                    "due_date": {
                        "type": "STRING"
                    }
                },

                "required": [
                    "task",
                    "owner",
                    "due_date"
                ]
            }
        },

        "meeting_status": {

            "type": "STRING",

            "enum": [
                "Confirmed",
                "Tentative",
                "Cancelled"
            ]
        }
    },

    "required": [
        "meeting_title",
        "participants",
        "decisions",
        "action_items",
        "meeting_status"
    ]
}


# ============================================================
# 4. GEMINI API CALL
# ============================================================

response = client.models.generate_content(

    model="gemini-3.5-flash",

    contents=prompt,

    config=types.GenerateContentConfig(

        # Tell Gemini to return JSON
        response_mime_type="application/json",

        # Define the expected structure
        response_schema=response_schema,


        # Define model behavior
        system_instruction="""
        You are a meeting intelligence assistant.

        Extract information accurately from meeting notes.

        Rules:
        - Do not invent information.
        - Only use information present in the notes.
        - Preserve names accurately.
        - Extract every explicit action item.
        - Return only the requested structured information.
        """
    )
)


# ============================================================
# 5. VIEW RAW JSON
# ============================================================

print(response.text)

{"meeting_title":"Product Launch Planning Meeting","participants":["Akash","Priya","Rahul","Neha"],"decisions":["Launch the new AI resume screener on September 15","The launch date is confirmed"],"action_items":[{"task":"finalize the landing page","owner":"Priya","due_date":"September 5"},{"task":"complete the payment integration","owner":"Rahul","due_date":"September 8"},{"task":"prepare the launch email campaign","owner":"Neha","due_date":"September 10"},{"task":"review the final product demo","owner":"Akash","due_date":"September 12"}],"meeting_status":"Confirmed"}


In [ ]:
# ============================================================
# 6. CONVERT JSON → PYTHON DICTIONARY
# ============================================================

result = json.loads(response.text)
result

{'meeting_title': 'Product Launch Planning Meeting',
 'participants': ['Akash', 'Priya', 'Rahul', 'Neha'],
 'decisions': ['Launch the new AI resume screener on September 15',
  'The launch date is confirmed'],
 'action_items': [{'task': 'finalize the landing page',
   'owner': 'Priya',
   'due_date': 'September 5'},
  {'task': 'complete the payment integration',
   'owner': 'Rahul',
   'due_date': 'September 8'},
  {'task': 'prepare the launch email campaign',
   'owner': 'Neha',
   'due_date': 'September 10'},
  {'task': 'review the final product demo',
   'owner': 'Akash',
   'due_date': 'September 12'}],
 'meeting_status': 'Confirmed'}

In [ ]:
# ============================================================
# 7. USE THE STRUCTURED DATA
# ============================================================

print("\n" + "=" * 50)
print("MEETING SUMMARY")
print("=" * 50)

print("\nMeeting:")
print(result["meeting_title"])

print("\nParticipants:")
for person in result["participants"]:
    print("-", person)

print("\nDecisions:")
for decision in result["decisions"]:
    print("-", decision)

print("\nAction Items:")

for item in result["action_items"]:

    print(
        f"- {item['task']} | "
        f"Owner: {item['owner']} | "
        f"Due: {item['due_date']}"
    )

print("\nMeeting Status:")
print(result["meeting_status"])


MEETING SUMMARY

Meeting:
Product Launch Planning Meeting

Participants:
- Akash
- Priya
- Rahul
- Neha

Decisions:
- Launch the new AI resume screener on September 15
- The launch date is confirmed

Action Items:
- finalize the landing page | Owner: Priya | Due: September 5
- complete the payment integration | Owner: Rahul | Due: September 8
- prepare the launch email campaign | Owner: Neha | Due: September 10
- review the final product demo | Owner: Akash | Due: September 12

Meeting Status:
Confirmed


In [ ]:
# Meeting notes --> Gemini --> Structured JSON output

In [ ]:
# Expected:
# {
#   "meeting_title": "...",
#   "participants": [...],
#   "action_items": [...]
# }

In [ ]:
# But imagine Gemini returns:
#  {
#   "title": "...",
#   "people": [...],
#   "tasks": [...]
# }

In [ ]:
# Meeting Notes
#    ↓
# Schema
#    ↓
# Validation (?)
#    ↓
# Reliable Application

In [ ]:
# Pydantic :

In [ ]:
from pydantic import BaseModel

In [ ]:
# ============================================================
# 1. PYDANTIC MODEL
# ============================================================
#
# This is our OUTPUT CONTRACT.
#
# We expect:
#
# name → string
# age  → integer
# city → string

class Person(BaseModel):
    name: str
    age:  int
    city: str

In [ ]:
# ============================================================
# 2. ASK GEMINI FOR JSON
# ============================================================

response = client.models.generate_content(

    model="gemini-3.5-flash",

    contents="""
    Give me a fictional person's name, age and city.
    """,

    config=types.GenerateContentConfig(

        response_mime_type="application/json",

        response_schema={
            "type": "OBJECT",

            "properties": {

                "name": {
                    "type": "STRING"
                },

                "age": {
                    "type": "INTEGER"
                },

                "city": {
                    "type": "STRING"
                }
            },

            "required": [
                "name",
                "age",
                "city"
            ]
        }
    )
)

In [ ]:
# ============================================================
# 3. SEE GEMINI'S RESPONSE
# ============================================================

print(response.text)

{"name":"Evelyn Vance","age":29,"city":"Seattle"}


In [ ]:
# ============================================================
# 4. JSON → PYTHON DICTIONARY
# ============================================================

data = json.loads(response.text)

print(data)


{'name': 'Evelyn Vance', 'age': 29, 'city': 'Seattle'}


In [ ]:
# ============================================================
# 5. PYDANTIC VALIDATION
# ============================================================
#
# Convert the dictionary into our Pydantic model.

person = Person(**data)
person

Person(name='Evelyn Vance', age=29, city='Seattle')

In [ ]:
# ============================================================
# 6. USE THE VALIDATED OBJECT
# ============================================================

print("\nName:", person.name)
print("Age:", person.age)
print("City:", person.city)


Name: Evelyn Vance
Age: 29
City: Seattle


In [ ]:
# ============================================================
# 7. WHY PYDANTIC?
# ============================================================
#
# Suppose Gemini returns:
#
# {
#   "name": "Rahul",
#   "age": "twenty eight", (instead of 28)
#   "city": "Delhi"
# }
#
# Pydantic can detect that "age" is not a valid integer
# for our contract.
#
# ============================================================

In [ ]:
# ============================================================
# 8. INTENTIONALLY TEST INVALID DATA
# ============================================================

bad_data = {
    "name": "Rahul",
    "age": "twenty eight",
    "city": "Delhi"
}

try:

    person = Person(**bad_data)

except Exception as e:

    print("\nValidation Error:")
    print(e)


Validation Error:
1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='twenty eight', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


In [ ]:
# Exercise: Meeting → Project Plan

# Problem Statement

# A company wants to automatically convert unstructured meeting notes into a project-management-ready plan.

# Gemini should extract:

# Project name
# Project status
# Key decisions
# Action items
# Owner
# Due date
# Priority

# The output will be consumed by another Python application, so the output must follow a strict contract.

In [ ]:
# Input :
# meeting_notes = """
# Website Redesign Project — Weekly Meeting

# The redesign is currently on track.

# The team decided to launch the new website on September 30.

# Priya will complete the homepage design by September 10.
# Rahul will finish the backend API integration by September 15.
# Neha will conduct user testing by September 20.

# The API integration is critical for the launch.
# The team agreed that all critical issues must be resolved
# before September 25.
# """

In [ ]:
# Output :
# {
#     "project_name": "Website Redesign",
#     "status": "On Track",
#     "decisions": [
#         "Launch the new website on September 30",
#         "Resolve all critical issues before September 25"
#     ],
#     "action_items": [
#         {
#             "task": "Complete homepage design",
#             "owner": "Priya",
#             "due_date": "September 10",
#             "priority": "Medium"
#         },
#         {
#             "task": "Finish backend API integration",
#             "owner": "Rahul",
#             "due_date": "September 15",
#             "priority": "High"
#         },
#         {
#             "task": "Conduct user testing",
#             "owner": "Neha",
#             "due_date": "September 20",
#             "priority": "Medium"
#         }
#     ]
# }

In [ ]:
# Student Tasks

# Task 1 — Create the Input Contract
# Define:

# meeting_notes → string

# Task 2 — Create the Output Contract
# Decide the types for:

# project_name
# status
# decisions
# action_items

# And inside action_items:

# task
# owner
# due_date
# priority

# Task 3 — Create the Gemini response_schema
# Use:

# response_mime_type="application/json"

# and your response_schema.

# Task 4 — Add Pydantic Validation
# Create:

# class ActionItem(BaseModel):
#     ...

# and:

# class ProjectPlan(BaseModel):
#     ...

# Use Literal to restrict:

# status:
# On Track
# At Risk
# Delayed

# priority:
# Low
# Medium
# High

# Task 5 — Test
# Test the application with at least 2 different meeting notes.

# Bonus Challenge
# Add:
# confidence_score
# with a valid range:
# 0 → 1
# For example:
# confidence_score: float = Field(ge=0, le=1)